## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger
from datetime import datetime
from utils.pipeline_monitoring import log_pipeline_run

dataset_name = "fuel_type_dim"
layer = "gold"

start_time = datetime.now()

logger = get_project_logger("Gold_Dimension_fuel_type")
logger.info("--- Starting Gold Layer: Creating dim_fuel_type ---")

## Creating a gold dimension view

In [0]:
start_time = datetime.now()

try:
    logger.info("Step 1/2: Running SQL to create dim_fuel_type view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.dim_fuel_type AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY fuel_type_nm) AS fuel_type_key,
        MIN(fuel_type_cd) AS fuel_type_cd,
        fuel_type_nm
    FROM (
        SELECT
            'N/A' AS fuel_type_cd,
            fuel_type_nm
        FROM transport_lakehouse.silver.silver_vehicles_private
        WHERE fuel_type_nm IS NOT NULL

        UNION

        SELECT
            CAST(fuel_type_cd AS STRING) AS fuel_type_cd,
            fuel_type_nm
        FROM transport_lakehouse.silver.silver_vehicles_two_wheeled
        WHERE fuel_type_nm IS NOT NULL
    ) t
    WHERE fuel_type_nm IS NOT NULL
    GROUP BY fuel_type_nm
    """
    
    spark.sql(sql_query)
    logger.info("Step 1/2: View created successfully.")

    logger.info("Step 2/2: Performing Quality Check (Row Count)")
    dim_count = spark.table("transport_lakehouse.gold.dim_fuel_type").count()
    logger.info(f"Quality Check Passed: dim_fuel_type contains {dim_count} unique fuel types.")

    end_time = datetime.now()

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="SUCCESS",
        rows_ingested=dim_count,
        error_message=None
    )

except Exception as e:
    end_time = datetime.now()
    logger.error(f"Failed to create Gold Dimension: {str(e)}")

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="FAILED",
        rows_ingested=0,
        error_message=str(e)
    )

    raise

logger.info("--- Gold dim_fuel_type Process Completed ---")
